# Apply Hang Analysis

Diagnose a **hung or very slow** `terraform apply` from **`TF_LOG=json`** trace output.

Looks for:

- **Stuck apply** — apply started but not completing (with **open duration** to last log line)
- **Open ApplyResourceChange RPCs** — provider requests with no downstream response at end of log
- **Graph wait loops** — dependency graph spinning during apply
- **SDK retry / 404 loops** — rate limits or missing Genesys Cloud objects

```bash
export TF_LOG=json
export TF_LOG_PATH=apply-hang.log
# optional: GENESYSCLOUD_SDK_DEBUG=true GENESYSCLOUD_SDK_DEBUG_FORMAT=Json
terraform apply
export TERRAFORM_LOG_PATH=apply-hang.log
```

For completed applies use `apply/analysis.ipynb`. Run `whatisit.ipynb` first if unsure.


In [ ]:
import sys
from pathlib import Path

_nb_root = Path.cwd()
if (_nb_root.parent / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root.parent))
elif (_nb_root / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root))

import notebook_setup

notebook_setup.setup()

import matplotlib.pyplot as plt
import pandas as pd

import commonlib.config as cfg
import commonlib.gencharts as gencharts
import commonlib.prep_apply_data as prep_apply_data
import commonlib.prep_hang_data as hang


In [ ]:
TAIL_MINUTES = 5
MIN_REPEAT = 3
WORKFLOW = "apply"

c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)

classification, records, counters = hang.load_hang_scan(c.TERRAFORM_LOG_PATH, tail_minutes=TAIL_MINUTES)
if not records:
    raise ValueError("No JSON log lines found. Capture with TF_LOG=json and run whatisit.ipynb first.")

normalized_records = prep_apply_data.normalize_records(records)
df_apply, df_merged_apply, df_open_apply = prep_apply_data.apply_timing_dataframes(
    normalized_records
)

if not classification.is_apply and not counters.apply_starts:
    raise ValueError(
        "No apply activity found. Use apply/hang-analysis.ipynb on a TF_LOG apply capture, "
        "or run whatisit.ipynb to pick the right notebook."
    )

summary = hang.hang_summary_for_workflow(
    counters,
    tail_minutes=TAIL_MINUTES,
    workflow=WORKFLOW,
    min_count=MIN_REPEAT,
    classification=classification,
)


## Summary

In [ ]:
print(f"Parsed lines: {summary['parsed_lines']:,}")
if summary['duration_minutes'] is not None:
    print(f"Log span: {summary['duration_minutes']:.1f} minutes")
if summary['first_timestamp']:
    print(f"Time range: {summary['first_timestamp']} → {summary['last_timestamp']}")
print(f"Tail window: last {summary['tail_minutes']:.0f} minutes")
print()
print(f"Primary suspect: {summary['primary_summary']}")
if summary['primary_detail']:
    print(f"  {summary['primary_detail']}")
print()
print(
    f"Apply: {summary['apply_resources_started']:,} resources started, "
    f"{summary['apply_resources_completed']:,} completed, "
    f"{summary['open_apply_rpc_types']:,} open RPC types, "
    f"{summary['dag_wait_pairs']:,} dag wait pairs, "
    f"{summary['sdk_retry_endpoints']:,} retry endpoints, "
    f"{summary['sdk_404_endpoints']:,} 404 endpoints"
)
if summary['sdk_status_codes']:
    print(f"SDK status codes: {summary['sdk_status_codes']}")


## Apply timing

How long apply steps took before the hang, and **open applies** (started but not completed) measured to the last log line.

In [ ]:
print(
    f"Completed apply pairs: {len(df_merged_apply.dropna(subset=['end_timestamp'])):,}  |  "
    f"Open applies (no complete): {len(df_open_apply):,}"
)

if df_open_apply.empty:
    print("No open applies — every apply_start had a matching apply_complete.")
else:
    display(
        df_open_apply[
            [
                "resource",
                "resource_type",
                "action",
                "start_timestamp",
                "open_minutes",
                "run",
            ]
        ].head(20)
    )

In [ ]:
if df_merged_apply.dropna(subset=["time_diff_minutes"]).empty:
    print("No matched apply_start/apply_complete pairs to chart.")
else:
    gencharts.generate_duration_by_resource_type(df_merged_apply, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_apply, metric="average", top_n=10)

In [ ]:
if df_merged_apply.dropna(subset=["time_diff_minutes"]).empty:
    print("No matched apply_start/apply_complete pairs to list.")
else:
    display(
        df_merged_apply[
            [
                "resource",
                "resource_type",
                "action",
                "start_timestamp",
                "end_timestamp",
                "time_diff_minutes",
                "run",
            ]
        ]
        .dropna(subset=["time_diff_minutes"])
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )

## Ranked hang suspects

In [ ]:
pd.DataFrame([
    {"category": v.category, "summary": v.summary, "detail": v.detail, "score": v.score}
    for v in summary["verdicts"]
]) if summary["verdicts"] else "No strong hang patterns detected (try lowering MIN_REPEAT)."


## Stuck apply resources

In [ ]:
hang.apply_imbalance_dataframe(counters, min_gap=1).head(20)


## Open ApplyResourceChange RPCs

Provider RPC requests still open at end of log.

In [ ]:
hang.apply_rpc_open_dataframe(counters).head(20)


## Graph wait loops during apply

In [ ]:
hang.blocked_resources_dataframe(counters, min_count=MIN_REPEAT).head(20)


## SDK retry storms

Endpoints returning `invocation_retry_after` — rate limits or SDK backoff.

In [ ]:
df_retries = hang.sdk_retry_dataframe(counters, min_count=MIN_REPEAT)
df_retries.head(20) if not df_retries.empty else "No SDK retry storms found."


In [ ]:
if not df_retries.empty:
    plot_df = df_retries.head(15).sort_values("retry_responses")
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["method_url"], plot_df["retry_responses"], color="tab:red")
    plt.xlabel("responses with retry_after")
    plt.title("SDK retry storms")
    plt.tight_layout()


## SDK 404 not found

Missing Genesys Cloud objects — often causes endless retry.

In [ ]:
df_404 = hang.sdk_not_found_dataframe(counters, min_count=2)
df_404.head(20) if not df_404.empty else "No repeated SDK 404 responses found."


## Tail activity (last few minutes)

What the log was doing right before capture.

In [ ]:
hang.tail_messages_dataframe(counters, top_n=25)
